Identification of enterotypes and enterotype differential gut microbial changes

To identify the enterotype in our cohort, we used Partitioning Around Medoids (PAM) analysis based on the relative abundance of microbial genera and used the Calinski-Harabasz index to select the optimal number of clusters, as in a previously published study.

In [ ]:
#Enterotype identification
rm(list=ls())
library(tidyverse)
library(ggplot2)
library(ggsci)
library(dplyr)
library(patchwork)
library(ggpubr)
library(cluster)
library(clusterSim)
library(ade4)

count_file_name="E:/QZ/01_metag/00_Rawdata/01_metaphlan/merged_abundance_table_genus.txt"

data <- read.csv(count_file_name,sep="\t",
                     row.names=1,check.names=F)
data<-data/100

dist.JSD <- function(inMatrix, pseudocount=0.000001, ...) {
  KLD <- function(x,y) sum(x *log(x/y))
  JSD<- function(x,y) sqrt(0.5 * KLD(x, (x+y)/2) + 0.5 * KLD(y, (x+y)/2))
  matrixColSize <- length(colnames(inMatrix))
  matrixRowSize <- length(rownames(inMatrix))
  colnames <- colnames(inMatrix)
  resultsMatrix <- matrix(0, matrixColSize, matrixColSize)
  
  inMatrix = apply(inMatrix,1:2,function(x) ifelse (x==0,pseudocount,x))
  
  for(i in 1:matrixColSize) {
    for(j in 1:matrixColSize) { 
      resultsMatrix[i,j]=JSD(as.vector(inMatrix[,i]),
                             as.vector(inMatrix[,j]))
    }
  }
  colnames -> colnames(resultsMatrix) -> rownames(resultsMatrix)
  as.dist(resultsMatrix)->resultsMatrix
  attr(resultsMatrix, "method") <- "dist"
  return(resultsMatrix) 
}

##PAM
pam.clustering=function(x,k) { # x is a distance matrix and k the number of clusters
  require(cluster)
  cluster = pam(as.dist(x), k, diss=TRUE)$clustering
  return(cluster)
}
data.dist=dist.JSD(data)

data.cluster=pam.clustering(data.dist, 
                            k=2)

nclusters = index.G1(t(data), data.cluster, d = data.dist, centrotypes = "medoids") 
nclusters = NULL

  for(k in 1:10)
{ 
  if(k==1)
  {
    nclusters[k] = NA 
  }
  else
  {
    data.cluster_temp = pam.clustering(data.dist, k)
    nclusters[k] = index.G1(t(data), data.cluster_temp,  d = data.dist, centrotypes = "medoids")
  }
}
pdf("CH_index.pdf",height=4,width=5)
nclusters[1] = 0
plot(nclusters, type="b", xlab="k clusters", ylab="CH index",lwd = 2)
dev.off()

k_best = which(nclusters == max(nclusters), arr.ind = TRUE)
k_best
k_best=2
data.cluster=pam.clustering(data.dist, k = 2)

mean(silhouette(data.cluster, data.dist)[, 3])

obs.pca=dudi.pca(data.frame(t(data)), scannf=F, nf=2)
obs.bet=bca(obs.pca, fac=as.factor(as.vector(data.cluster)), scannf=F, nf=2) 
head(obs.bet$tab)
write.table(t(obs.bet$tab),"bca_table_drivers.txt",sep = '\t',quote=F)

obs.pcoa=dudi.pco(data.dist, scannf=F, nf=2)

write.csv(data.cluster,"data.cluster.csv",quote=F) 
write.csv(obs.pcoa$li,"obs.pcoa_li.csv",quote=F) 

For enterotype differential analysis, MaAsLin2 (version 1.16.0) was employed. Age, sex, and BMI were included as covariates for correction, and enterotype was fitted into the model. 
Feature ≈ enterotype + sex + age + BMI
A false discovery rate (FDR, q value) less than 0.05 was set as the threshold for statistical significance. The details of enterotype-related differential species and functions are listed in Supplementary Table 2.

In [ ]:
#enterotype differential gut microbial changes
rm(list=ls())
library(tidyverse)
library(ggplot2)
library(ggsci)
library(dplyr)
library(patchwork)
library(Maaslin2)

count_file_name="E:/QZ/QZ_metagenomics/01_Upstream/04_Result/01_metaphlan/merged_abundance_table_read_count_species.txt"
df_input_data = read.table(file             = count_file_name,
                           header           = TRUE,
                           sep              = "\t", 
                           row.names        = 1,
                           stringsAsFactors = FALSE,
                           check.names = F)
df_input_data[1:5, 1:5]

df_input_metadata = read.table(file             = "E:/QZ/QZ_metagenomics/04_Merge/Sample_metadata.csv", 
                               header           = TRUE, 
                               sep              = ",", 
                               row.names        = 1,
                               stringsAsFactors = FALSE)
rownames(df_input_metadata)

table(df_input_metadata$Group)
df_input_metadata<-df_input_metadata%>%filter(Group=="Control")

Enterotype_cluster = read.csv(
  "E:/QZ/QZ_metagenomics/04_Merge/02_Read_based/01_Taxonomy/Enterotype/2/data.cluster.csv")
names(Enterotype_cluster)[2]<-"Enterotype"
table(Enterotype_cluster$Enterotype)

df_input_metadata<-merge(Enterotype_cluster,df_input_metadata,
            by.x = "X", by.y = "Name_metaphlan")

rownames(df_input_metadata)<-df_input_metadata$X
table(df_input_metadata$Group)
table(df_input_metadata$Sex)
table(df_input_metadata$Batch)
intersect(rownames(df_input_metadata),colnames(df_input_data))
df_input_data<-df_input_data[,intersect(rownames(df_input_metadata),colnames(df_input_data))]
head(df_input_metadata)
colnames(df_input_metadata)
table(df_input_metadata$Sex)
fit_data = Maaslin2(input_data     = df_input_data,
                    input_metadata = df_input_metadata,
                    min_prevalence = 0.05,
                    # normalization  = "NONE",
                    # standardize = FALSE,
                    output         = "Result", 
                    # cores = 4 ,
                    fixed_effects  = c("Age",
                                       "Sex",
                                       "Enterotype",
                                       "BMI"))
